# Pilot v3: Linguistic Complexity + ERC

Νέο: DC/C proxy + ADD proxy (syntactic metrics χωρίς parser)

**Cells με τη σειρά:**
1. Install
2. Imports
3. Mount Drive
4. Paths
5. Load model
6. Build dataset
7. Sampling
8. Complexity metrics
9. Composite scores
10. Inference
11. Correlation
12. Plots
13. Save

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn seaborn

In [ ]:
import os, re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
LABELS   = ['neutral', 'joy', 'sadness', 'anger', 'surprise', 'fear', 'disgust']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
SEED     = 42
np.random.seed(SEED)

DIALOG_COL  = 'Dialogue_ID'
UTTID_COL   = 'Utterance_ID'
SPEAKER_COL = 'Speaker'
TEXT_COL    = 'Utterance'
LABEL_COL   = 'Emotion'
MAX_LEN     = 512

print('Device:', DEVICE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ΑΛΛΑΞΕ ΕΔΩ
CHECKPOINT_PATH = '/content/drive/MyDrive/epoch_checkpoints_seed42_emoberta_roberta_with_spaces/roberta_meld_final_seed42_BEST'
MELD_TEST_CSV   = '/content/drive/MyDrive/test_sent_emo.csv'
OUTPUT_DIR      = '/content/drive/MyDrive/pilot_results_v3'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# None = full dataset | integer = stratified sample
N_SAMPLES = None

print('Checkpoint:', os.path.exists(CHECKPOINT_PATH))
print('CSV:       ', os.path.exists(MELD_TEST_CSV))

In [ ]:
print('Loading model...')
tok   = AutoTokenizer.from_pretrained(CHECKPOINT_PATH, use_fast=True, add_prefix_space=True)
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_PATH)
model.eval().to(DEVICE)
print('Model loaded on', DEVICE)
print('id2label:', model.config.id2label)

In [ ]:
def build_context_dataset(df, tokenizer, max_length=512, speaker_caps=True):
    df = df.copy()
    df[TEXT_COL]    = df[TEXT_COL].astype(str)
    df[SPEAKER_COL] = df[SPEAKER_COL].astype(str)
    df[LABEL_COL]   = df[LABEL_COL].astype(str).str.strip().str.lower()
    df[UTTID_COL]   = pd.to_numeric(df[UTTID_COL], errors='coerce')
    df = df.dropna(subset=[UTTID_COL]).copy()
    df[UTTID_COL]   = df[UTTID_COL].astype(int)
    df = df[df[LABEL_COL].isin(LABELS)].copy()
    df = df.sort_values([DIALOG_COL, UTTID_COL]).reset_index(drop=True)

    cls_id     = tokenizer.cls_token_id
    sep_id     = tokenizer.sep_token_id
    max_tokens = max_length - 2
    space_ids  = tokenizer.encode(' ', add_special_tokens=False)
    if len(space_ids) == 0:
        space_ids = []

    all_ids, all_attn, all_labels = [], [], []
    all_texts, all_dialog, all_turn = [], [], []

    for d_id, g in df.groupby(DIALOG_COL, sort=False):
        speakers = g[SPEAKER_COL].tolist()
        utts     = g[TEXT_COL].tolist()
        labs     = g[LABEL_COL].tolist()
        turns    = g[UTTID_COL].tolist()
        if speaker_caps:
            speakers = [s.upper() for s in speakers]
        seg_text = [f'{s}: {u}' for s, u in zip(speakers, utts)]
        seg_ids  = [tokenizer.encode(x, add_special_tokens=False) for x in seg_text]
        n = len(seg_ids)

        for t in range(n):
            target_ids = seg_ids[t][:]
            if len(target_ids) + 2 > max_tokens:
                target_ids = target_ids[:max(0, max_tokens - 2)]
            seq_ids  = [sep_id] + target_ids + [sep_id]
            seq_text = ' </s> ' + seg_text[t] + ' </s> '
            left, right = t - 1, t + 1
            bl = br = False
            while True:
                changed = False
                if left >= 0 and not bl:
                    cand = seg_ids[left]
                    need = len(cand) + (len(space_ids) if space_ids else 0)
                    if len(seq_ids) + need <= max_tokens:
                        seq_ids  = cand + space_ids + seq_ids if space_ids else cand + seq_ids
                        seq_text = seg_text[left] + ' ' + seq_text
                        left -= 1; changed = True
                    else:
                        bl = True
                if right < n and not br:
                    cand = seg_ids[right]
                    need = len(cand) + (len(space_ids) if space_ids else 0)
                    if len(seq_ids) + need <= max_tokens:
                        seq_ids  = seq_ids + space_ids + cand if space_ids else seq_ids + cand
                        seq_text = seq_text + ' ' + seg_text[right]
                        right += 1; changed = True
                    else:
                        br = True
                if not changed:
                    break
            input_ids = ([cls_id] + seq_ids + [sep_id])[:max_length]
            all_ids.append(input_ids)
            all_attn.append([1] * len(input_ids))
            all_labels.append(label2id[labs[t]])
            all_texts.append('<s> ' + seq_text.strip() + ' </s>')
            all_dialog.append(d_id)
            all_turn.append(turns[t])

    return Dataset.from_dict({
        'dialogue_id':      all_dialog,
        'utterance_id':     all_turn,
        'context_text_raw': all_texts,
        'input_ids':        all_ids,
        'attention_mask':   all_attn,
        'labels':           all_labels,
    })

print('Loading MELD test CSV...')
test_df = pd.read_csv(MELD_TEST_CSV)
print('Rows:', len(test_df))

print('Building context dataset...')
test_ds = build_context_dataset(test_df, tok, MAX_LEN)

df_full = pd.DataFrame({
    'dialogue_id':      test_ds['dialogue_id'],
    'utterance_id':     test_ds['utterance_id'],
    'label_id':         test_ds['labels'],
    'label':            [id2label[i] for i in test_ds['labels']],
    'context_text_raw': test_ds['context_text_raw'],
    'input_ids':        test_ds['input_ids'],
    'attention_mask':   test_ds['attention_mask'],
})
print('Dataset:', len(df_full), 'utterances')
print(df_full['label'].value_counts())

In [ ]:
if N_SAMPLES is None:
    df = df_full.copy().reset_index(drop=True)
    print('Full dataset:', len(df), 'utterances')
else:
    n_per_class = N_SAMPLES // len(LABELS)
    sampled = []
    for lbl in LABELS:
        sub = df_full[df_full['label'] == lbl]
        sampled.append(sub.sample(min(n_per_class, len(sub)), random_state=SEED))
    df = pd.concat(sampled).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print('Stratified sample:', len(df), 'utterances')
print(df['label'].value_counts())

In [ ]:
# ── Complexity metrics ──

def clean_text(text):
    text = re.sub(r'</?s>', ' ', str(text))
    text = re.sub(r'[A-Z]{2,}:\s*', '', text)
    return re.sub(r'\s+', ' ', text).strip()

# Surface
def compute_ari(text):
    words = text.split(); n_w = len(words)
    if n_w == 0: return 0.0
    n_c = sum(len(w) for w in words)
    n_s = max(1, text.count('.')+text.count('!')+text.count('?'))
    return round(4.71*(n_c/n_w) + 0.5*(n_w/n_s) - 21.43, 3)

def compute_lix(text):
    words = text.split(); n_w = len(words)
    if n_w == 0: return 0.0
    n_s = max(1, text.count('.')+text.count('!')+text.count('?'))
    lw  = sum(1 for w in words if len(re.sub(r'[^a-zA-Z]','',w)) > 6)
    return round((n_w/n_s) + 100*(lw/n_w), 3)

def compute_mattr(text, window=50):
    tokens = text.lower().split(); n = len(tokens)
    if n < window: return round(len(set(tokens))/max(1,n), 3)
    return round(float(np.mean([len(set(tokens[i:i+window]))/window
                                for i in range(n-window+1)])), 3)

# Syntactic proxies
SUB_MARKERS = {'because','although','since','while','if','when','that',
               'which','who','unless','until','after','before','though',
               'whenever','wherever','whether','whereas','even'}
COORD_MARKERS = {'and','but','or','nor','so','yet','for'}

def compute_dc_c_proxy(text):
    """DC/C proxy — Lu (2010) via subordinating markers"""
    words = [w.lower() for w in re.findall(r"[a-zA-Z']+", text)]
    if not words: return 0.0
    dc    = sum(1 for w in words if w in SUB_MARKERS)
    total = max(1, text.count('.')+text.count('!')+text.count('?') +
               sum(1 for w in words if w in SUB_MARKERS | COORD_MARKERS))
    return round(dc / total, 3)

STOP_WORDS = {'the','a','an','is','are','was','were','be','been','being',
              'have','has','had','do','does','did','will','would','shall',
              'should','may','might','must','can','could','to','of','in',
              'on','at','by','for','with','as','from','and','but','or',
              'so','yet','nor','this','that','it','its','i','we','you',
              'he','she','they','my','your','his','her','our','their'}

def compute_add_proxy(text):
    """ADD proxy — Liu (2008) via content word distance"""
    words   = [w.lower() for w in text.split() if re.sub(r'[^a-zA-Z]','',w)]
    if len(words) < 2: return 0.0
    content = [(i,w) for i,w in enumerate(words) if w not in STOP_WORDS]
    if len(content) < 2: return 0.0
    dists = [content[i+1][0]-content[i][0] for i in range(len(content)-1)]
    return round(sum(dists)/len(dists), 3)

# Compute
print('Computing complexity metrics...')
cleaned = df['context_text_raw'].apply(clean_text)
df['ARI']        = cleaned.apply(compute_ari)
df['LIX']        = cleaned.apply(compute_lix)
df['MATTR']      = cleaned.apply(compute_mattr)
df['DC_C_proxy'] = cleaned.apply(compute_dc_c_proxy)
df['ADD_proxy']  = cleaned.apply(compute_add_proxy)
df['n_words']    = cleaned.apply(lambda t: len(t.split()))

print('Done')
display(df[['ARI','LIX','MATTR','DC_C_proxy','ADD_proxy','n_words']].describe().round(3))

In [ ]:
# ── Composite scores ──

def normalize(s):
    mn,mx = s.min(),s.max()
    return (s-mn)/(mx-mn) if mx!=mn else pd.Series([0.5]*len(s),index=s.index)

df['ARI_n']     = normalize(df['ARI'])
df['LIX_n']     = normalize(df['LIX'])
df['MATTR_inv'] = 1 - normalize(df['MATTR'])
df['DC_n']      = normalize(df['DC_C_proxy'])
df['ADD_n']     = normalize(df['ADD_proxy'])

# Surface only
df['composite_surface'] = (
    0.40*df['ARI_n'] + 0.35*df['LIX_n'] + 0.25*df['MATTR_inv']
).round(4)

# Full with syntactic
df['composite_full'] = (
    0.30*df['DC_n']   +
    0.25*df['ADD_n']  +
    0.25*df['ARI_n']  +
    0.20*df['MATTR_inv']
).round(4)

print('composite_surface:', df['composite_surface'].describe().round(3).to_dict())
print('composite_full:   ', df['composite_full'].describe().round(3).to_dict())
print('Correlation between composites:', round(df['composite_surface'].corr(df['composite_full']),3))

In [ ]:
# ── EmoBERTa Inference ──

@torch.no_grad()
def run_inference(input_ids_list, attn_list, model, device, batch_size=16):
    results = []
    for i in range(0, len(input_ids_list), batch_size):
        batch_ids  = input_ids_list[i: i+batch_size]
        batch_attn = attn_list[i: i+batch_size]
        max_len    = max(len(x) for x in batch_ids)
        pad_id     = 1
        padded_ids = torch.tensor(
            [x+[pad_id]*(max_len-len(x)) for x in batch_ids], dtype=torch.long
        ).to(device)
        padded_attn = torch.tensor(
            [x+[0]*(max_len-len(x)) for x in batch_attn], dtype=torch.long
        ).to(device)
        probs = torch.softmax(
            model(input_ids=padded_ids, attention_mask=padded_attn).logits, dim=-1
        ).cpu().numpy()
        for p in probs:
            pid = int(np.argmax(p))
            results.append({
                'predicted_id':    pid,
                'predicted_label': id2label[pid],
                'confidence':      round(float(p[pid]), 4),
            })
        if (i // batch_size) % 5 == 0:
            print(f'  {min(i+batch_size, len(input_ids_list))}/{len(input_ids_list)}')
    return pd.DataFrame(results)

print('Running inference...')
inf_df = run_inference(
    df['input_ids'].tolist(),
    df['attention_mask'].tolist(),
    model, DEVICE
)
df['predicted_id']    = inf_df['predicted_id'].values
df['predicted_label'] = inf_df['predicted_label'].values
df['confidence']      = inf_df['confidence'].values
df['is_correct']      = (df['predicted_label'] == df['label']).astype(int)

print(f'Accuracy:    {df["is_correct"].mean():.3f}')
print(f'Weighted F1: {f1_score(df["label_id"], df["predicted_id"], average="weighted"):.3f}')
print(f'Macro F1:    {f1_score(df["label_id"], df["predicted_id"], average="macro"):.3f}')

In [ ]:
# ── Correlation Analysis ──

complexity_cols = ['ARI','LIX','MATTR',
                   'DC_C_proxy','ADD_proxy',
                   'composite_surface','composite_full']
target_cols = ['confidence','is_correct']

print('='*70)
print('PEARSON CORRELATION — ALL METRICS')
print('='*70)
corr_rows = []
for comp in complexity_cols:
    for tgt in target_cols:
        r,p = stats.pearsonr(df[comp], df[tgt])
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
        marker = '  <- SIGNAL' if abs(r)>0.1 and p<0.05 else ''
        print(f'  {comp:22s} vs {tgt:12s}  r={r:+.3f}  p={p:.4f}  {sig}{marker}')
        corr_rows.append({'complexity':comp,'target':tgt,
                          'r':round(r,3),'p':round(p,4),'sig':sig})
corr_df = pd.DataFrame(corr_rows)
display(corr_df)

In [ ]:
# ── Complexity bins ──
bins   = ['Low','Medium-Low','Medium-High','High']
colors = ['#1D9E75','#378ADD','#BA7517','#D85A30']

df['bin_surface'] = pd.qcut(df['composite_surface'], q=4, labels=bins)
df['bin_full']    = pd.qcut(df['composite_full'],    q=4, labels=bins)

print('Bins created OK')
print(df['bin_surface'].value_counts().sort_index())
print(df['bin_full'].value_counts().sort_index())

In [ ]:
# ── RQ1: F1 ανά complexity bin ──
fig, axes = plt.subplots(1,2,figsize=(14,4))
for ax, bin_col, title in zip(
    axes,
    ['bin_surface','bin_full'],
    ['Surface only (ARI+LIX+MATTR)','Full (+ DC/C proxy + ADD proxy)']
):
    bin_f1 = {}
    for b in bins:
        sub = df[df[bin_col]==b]
        if len(sub)==0: continue
        bin_f1[b] = round(f1_score(sub['label_id'],sub['predicted_id'],
                                    average='weighted',zero_division=0),3)
    bars = ax.bar(bin_f1.keys(),bin_f1.values(),color=colors,width=0.6,edgecolor='white')
    for bar,val in zip(bars,bin_f1.values()):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,
                f'{val:.3f}',ha='center',va='bottom',fontsize=10)
    ax.set_xlabel('Complexity Group'); ax.set_ylabel('Weighted F1')
    ax.set_title(f'RQ1 — {title}',fontsize=11); ax.set_ylim(0,1.0)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/rq1_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rq1_f1_comparison.png')

In [ ]:
# ── RQ2: Emotion x Complexity heatmap ──
fig, axes = plt.subplots(1,2,figsize=(16,5))
for ax, bin_col, title in zip(
    axes,
    ['bin_surface','bin_full'],
    ['Surface composite','Full composite']
):
    matrix = np.full((len(LABELS),len(bins)), np.nan)
    for j,b in enumerate(bins):
        sub = df[df[bin_col]==b]
        for i,emo in enumerate(LABELS):
            mask = sub['label']==emo
            if mask.sum()<3: continue
            matrix[i,j] = round(f1_score(sub[mask]['label_id'],sub[mask]['predicted_id'],
                                          average='weighted',zero_division=0),2)
    sns.heatmap(matrix,annot=True,fmt='.2f',
                xticklabels=bins,yticklabels=LABELS,
                cmap='RdYlGn',vmin=0,vmax=1,linewidths=0.5,ax=ax)
    ax.set_title(f'RQ2 Blind Spots — {title}',fontsize=11)
    ax.set_xlabel('Complexity Group'); ax.set_ylabel('Emotion')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/rq2_emotion_heatmap_comparison.png', dpi=150)
plt.show()
print('Saved: rq2_emotion_heatmap_comparison.png')

In [ ]:
# ── Scatter plots ──
fig, axes = plt.subplots(2,2,figsize=(12,8))
pairs = [
    ('composite_surface','confidence','#378ADD'),
    ('composite_full','confidence','#7F77DD'),
    ('DC_C_proxy','confidence','#1D9E75'),
    ('ADD_proxy','confidence','#BA7517'),
]
for ax,(metric,tgt,color) in zip(axes.flatten(),pairs):
    x,y = df[metric],df[tgt]
    r,p = stats.pearsonr(x,y)
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
    ax.scatter(x,y,alpha=0.2,s=8,color=color)
    xs = np.linspace(x.min(),x.max(),100)
    m,b = np.polyfit(x,y,1)
    ax.plot(xs,m*xs+b,color='crimson',lw=1.5)
    ax.set_xlabel(metric,fontsize=10)
    ax.set_ylabel('Confidence',fontsize=10)
    ax.set_title(f'{metric}\nr={r:+.3f} {sig}',fontsize=10)

plt.suptitle('Complexity Metrics vs EmoBERTa Confidence',fontsize=12,y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/scatter_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: scatter_all_metrics.png')

In [ ]:
# ── Correlation heatmap ──
corr_pivot = corr_df.pivot(index='complexity',columns='target',values='r')
fig,ax = plt.subplots(figsize=(6,6))
sns.heatmap(corr_pivot,annot=True,fmt='.3f',
            cmap='coolwarm',center=0,vmin=-0.3,vmax=0.3,
            linewidths=0.5,ax=ax)
ax.set_title('Pearson r: All Complexity Metrics vs Model Behavior',fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/correlation_heatmap_all.png', dpi=150)
plt.show()
print('Saved: correlation_heatmap_all.png')

In [ ]:
# ── Save results ──
save_cols = [
    'dialogue_id','utterance_id','label','label_id',
    'ARI','LIX','MATTR','DC_C_proxy','ADD_proxy',
    'composite_surface','composite_full','n_words',
    'predicted_label','predicted_id','confidence','is_correct',
    'bin_surface','bin_full'
]
df[save_cols].to_csv(f'{OUTPUT_DIR}/results_v3.csv', index=False)
corr_df.to_csv(f'{OUTPUT_DIR}/correlation_v3.csv', index=False)

print('Saved to:', OUTPUT_DIR)
print()
print('SUMMARY')
print(f'  Utterances:  {len(df)}')
print(f'  Accuracy:    {df["is_correct"].mean():.3f}')
print(f'  Weighted F1: {f1_score(df["label_id"],df["predicted_id"],average="weighted"):.3f}')
print(f'  Macro F1:    {f1_score(df["label_id"],df["predicted_id"],average="macro"):.3f}')
print()
print('Syntactic metrics:')
display(corr_df[corr_df['complexity'].isin(['DC_C_proxy','ADD_proxy'])]
        [['complexity','target','r','p','sig']])
print()
print('Composite full:')
display(corr_df[corr_df['complexity']=='composite_full'][['target','r','p','sig']])
!ls -lh "$OUTPUT_DIR"